In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tanjimshaharia/human-articles-100/human_articles_100.csv


In [2]:
# ============================================================
# STAGE 2: LLM-GENERATED BANGLA TEXT USING GEMINI API
# ============================================================

!pip -q install -U google-genai pandas tqdm

import os
import re
import time
import pandas as pd
from tqdm.auto import tqdm
from google import genai
from google.genai import types

# ============================================================
# 1. CONFIG
# ============================================================

# IMPORTANT:
# Do NOT paste your API key directly into this notebook.
# Add it as a Kaggle Secret named: GEMINI_API_KEY
#
# Kaggle:
# Notebook -> Add-ons -> Secrets -> Add new secret
#
# Name:
# GEMINI_API_KEY
#
# Then enable the secret for this notebook.

API_KEY = os.environ.get("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. "
        "Add your Gemini API key as a Kaggle Secret named GEMINI_API_KEY."
    )

client = genai.Client(api_key=API_KEY)

# Use a current Gemini model available to your API account.
MODEL = "gemini-2.5-flash"

INPUT_FILE = "/kaggle/input/datasets/tanjimshaharia/human-articles-100/human_articles_100.csv"

OUT_FILE = "/kaggle/working/llm_bangla_articles.csv"
FAILED_FILE = "/kaggle/working/llm_failed_topics.csv"

MIN_WORDS = 100
MAX_WORDS = 200
TARGET_WORDS = 150

MAX_RETRIES = 5

# Fixed generation settings for consistency
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_OUTPUT_TOKENS = 500

# Small delay between requests to reduce rate-limit problems
SLEEP_SECONDS = 1.0


# ============================================================
# 2. LOAD HUMAN DATASET
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("Loaded:", len(df), "topics")
print("Columns:")
print(df.columns.tolist())

required_cols = ["topic_id", "topic"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Remove accidental duplicate topic IDs
df = df.drop_duplicates(subset=["topic_id"]).copy()

print("Unique topics:", len(df))


# ============================================================
# 3. BANGLA VALIDATION
# ============================================================

BANGLA_RE = re.compile(r"[\u0980-\u09FF]")

BAD_PATTERNS = re.compile(
    r"(https?://|www\.|read more|subscribe|advertisement|"
    r"©|সূত্র:|ছবি:|ফাইল ছবি|প্রতিবেদক:)",
    re.I
)

def word_count(text):
    return len(re.findall(r"\S+", text))


def bangla_ratio(text):
    chars = re.sub(r"\s+", "", text)

    if not chars:
        return 0.0

    bangla_chars = len(BANGLA_RE.findall(chars))

    return bangla_chars / len(chars)


def clean_output(text):
    """
    Clean Gemini output without rewriting its content.
    """

    if not isinstance(text, str):
        return ""

    text = text.strip()

    # Remove markdown fences if Gemini adds them
    text = re.sub(r"^```(?:text|bangla|bn)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    # Remove accidental labels
    text = re.sub(
        r"^(বাংলা লেখা|লেখা|উত্তর|অনুচ্ছেদ)\s*:\s*",
        "",
        text,
        flags=re.I
    )

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def validate_text(text):

    if not text:
        return False, "empty"

    wc = word_count(text)

    if wc < MIN_WORDS:
        return False, f"too_short_{wc}"

    if wc > MAX_WORDS:
        return False, f"too_long_{wc}"

    br = bangla_ratio(text)

    if br < 0.65:
        return False, f"low_bangla_ratio_{br:.2f}"

    if BAD_PATTERNS.search(text):
        return False, "bad_pattern"

    # Should contain at least some sentence punctuation
    if not re.search(r"[।!?]", text):
        return False, "no_sentence_punctuation"

    return True, "OK"


# ============================================================
# 4. GENERATION PROMPT
# ============================================================

def make_prompt(topic):

    return f"""
আপনি একজন পেশাদার বাংলা সংবাদ লেখক।

নিচের বিষয়টি নিয়ে একটি সম্পূর্ণ নতুন, মৌলিক বাংলা সংবাদধর্মী অনুচ্ছেদ লিখুন।

বিষয়:
{topic}

নির্দেশনা:

1. শুধুমাত্র বাংলায় লিখবেন।
2. প্রায় ১৫০টি শব্দ লিখবেন।
3. মোট শব্দসংখ্যা ১০০ থেকে ২০০-এর মধ্যে রাখবেন।
4. স্বাভাবিক, পরিষ্কার এবং প্রমিত বাংলা ব্যবহার করবেন।
5. সংবাদ প্রতিবেদনধর্মী নিরপেক্ষ ভাষা ব্যবহার করবেন।
6. বিষয়টির ওপর তথ্যভিত্তিক সাধারণ সংবাদধর্মী লেখা তৈরি করবেন।
7. কোনো শিরোনাম লিখবেন না।
8. কোনো bullet point বা numbered list ব্যবহার করবেন না।
9. কোনো markdown ব্যবহার করবেন না।
10. কোনো URL, citation, source attribution বা reference লিখবেন না।
11. এই লেখাটি সম্পূর্ণ নতুনভাবে তৈরি করবেন। কোনো বিদ্যমান লেখা অনুকরণ বা পুনর্লিখন করবেন না।
12. শুধু চাওয়া বাংলা অনুচ্ছেদটি লিখবেন। অন্য কোনো ব্যাখ্যা দেবেন না।

বিষয়টি অবশ্যই লেখাটির মূল বিষয় হতে হবে।
"""


# ============================================================
# 5. GENERATE ONE TEXT
# ============================================================

def generate_one(topic):

    last_error = ""

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = client.models.generate_content(
                model=MODEL,
                contents=make_prompt(topic),
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                )
            )

            text = clean_output(response.text or "")

            ok, reason = validate_text(text)

            if ok:
                return text, "OK", attempt

            last_error = reason

            print(
                f"  Attempt {attempt}: rejected -> {reason}"
            )

        except Exception as e:

            last_error = str(e)

            print(
                f"  Attempt {attempt}: API error -> {e}"
            )

        time.sleep(2 * attempt)

    return "", last_error, MAX_RETRIES


# ============================================================
# 6. GENERATE ALL TOPICS
# ============================================================

results = []
failed = []

print("\nStarting Gemini generation...")
print("Topics:", len(df))
print("=" * 60)

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Generating LLM Bangla"
):

    topic_id = row["topic_id"]
    topic = str(row["topic"]).strip()

    text, status, attempts = generate_one(topic)

    if text:

        results.append({
            "topic_id": topic_id,
            "topic": topic,
            "llm_bangla": text,
            "llm_word_count": word_count(text),
            "llm_bangla_ratio": round(bangla_ratio(text), 4),
            "llm_model": MODEL,
            "llm_temperature": TEMPERATURE,
            "llm_status": status,
            "generation_attempts": attempts,
        })

    else:

        failed.append({
            "topic_id": topic_id,
            "topic": topic,
            "status": status,
            "attempts": attempts,
        })

    time.sleep(SLEEP_SECONDS)


# ============================================================
# 7. SAVE RESULTS
# ============================================================

llm_df = pd.DataFrame(results)
failed_df = pd.DataFrame(failed)

llm_df.to_csv(
    OUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

failed_df.to_csv(
    FAILED_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 8. FINAL REPORT
# ============================================================

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

print(f"Requested topics : {len(df)}")
print(f"Successful       : {len(llm_df)}")
print(f"Failed           : {len(failed_df)}")

if len(llm_df):
    print(
        f"Average words   : "
        f"{llm_df['llm_word_count'].mean():.1f}"
    )

print("\nOutput:")
print(OUT_FILE)

print("\nFailed topics:")
print(FAILED_FILE)

if len(failed_df):

    print("\nTopics that need retry:")
    print(
        failed_df[
            ["topic_id", "topic", "status"]
        ].to_string(index=False)
    )

else:

    print("\n🔥 ALL TOPICS GENERATED SUCCESSFULLY 🔥")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-

ValueError: GEMINI_API_KEY not found. Add your Gemini API key as a Kaggle Secret named GEMINI_API_KEY.